# CGC AMX Notebook

Documentation and manual test notebook for the CGC `AMX` wrapper.

Structure:
- one simple end-to-end sequence
- one connected-session section for wrapper methods that do not require `load_config()`
- one configured-session section for timing and pulser helpers
- one low-level transport section for `open_port()` / `close_port()` style calls


In [ ]:
from cgc.amx import AMX

DEVICE_ID = "AMX_TEST"
COM_PORT = 8
PORT_NUMBER = 0
BAUDRATE = 230400
STANDBY_CONFIG_NUMBER = None  # Optional: replace with a validated disabled config.
OPERATING_CONFIG_NUMBER = None  # Optional: replace with a validated operating config from list_configs().
CONFIG_NUMBER = 0  # Replace with an index reported by list_configs() for config read/write checks.
TIMEOUT_S = 5.0

PULSER_NO = 0
SWITCH_NO = 0
FREQUENCY_HZ = 2000.0
FREQUENCY_KHZ = 2.0
DUTY_CYCLE = 0.5
PULSER_DELAY_TICKS = 100
PULSER_WIDTH_TICKS = 100
SWITCH_ENABLE_DELAY_TICKS = 10
SWITCH_TRIGGER_RISE_TICKS = 10
SWITCH_TRIGGER_FALL_TICKS = 10

## 1. Simple Sequence

Recommended high-level sequence: instantiate, run `initialize()` to connect,
optionally load validated standby and/or operating configs, tweak one pulser
setting, read back the oscillator frequency, then shutdown. When no explicit
config is supplied, `initialize()` will still connect and will auto-load a
valid `Standby` config into controller memory when the AMX exposes one.


In [ ]:
amx_simple = AMX(device_id=DEVICE_ID, com=COM_PORT)  # Optional arguments: port=PORT_NUMBER, baudrate=BAUDRATE
startup_kwargs = {"timeout_s": TIMEOUT_S}
if OPERATING_CONFIG_NUMBER is not None:
    startup_kwargs["operating_config"] = OPERATING_CONFIG_NUMBER
if STANDBY_CONFIG_NUMBER is not None:
    startup_kwargs["standby_config"] = STANDBY_CONFIG_NUMBER
startup_state = amx_simple.initialize(**startup_kwargs)
print("startup_state", startup_state)

In [ ]:
amx_simple.set_frequency_hz(FREQUENCY_HZ)
amx_simple.set_pulser_duty_cycle(PULSER_NO, DUTY_CYCLE)
print("amx_simple.get_frequency_hz()", amx_simple.get_frequency_hz())

In [ ]:
amx_simple.shutdown()  # shutdown sequence and disconnects from the instrument.
amx_simple.close()  # releases Python-side resources

## 2. Connected Session

Use a dedicated instance to exercise connected-state wrapper methods one by one.
`connect()` is the intended entry point here.


In [ ]:
amx = AMX(
    device_id=f"{DEVICE_ID}_connected",
    com=COM_PORT,
    port=PORT_NUMBER,
    baudrate=BAUDRATE,
)

In [ ]:
print("amx.get_status()", amx.get_status())

In [ ]:
print("amx.connect(timeout_s=TIMEOUT_S)", amx.connect(timeout_s=TIMEOUT_S))

In [ ]:
print("amx.get_status()", amx.get_status())

In [ ]:
print("amx.format_status(amx.NO_ERR)", amx.format_status(amx.NO_ERR))

In [ ]:
print("amx.describe_error(amx.NO_ERR)", amx.describe_error(amx.NO_ERR))

In [ ]:
print("amx.get_product_info()", amx.get_product_info())

In [ ]:
print("amx.list_configs()", amx.list_configs())

In [ ]:
print("amx.get_main_state()", amx.get_main_state())

In [ ]:
print("amx.get_device_state()", amx.get_device_state())

In [ ]:
print("amx.get_controller_state()", amx.get_controller_state())

In [ ]:
print("amx.get_housekeeping()", amx.get_housekeeping())

In [ ]:
print("amx.get_sensor_data()", amx.get_sensor_data())

In [ ]:
print("amx.get_fan_data()", amx.get_fan_data())

In [ ]:
print("amx.get_led_data()", amx.get_led_data())

In [ ]:
print("amx.get_cpu_data()", amx.get_cpu_data())

In [ ]:
print("amx.get_uptime()", amx.get_uptime())

In [ ]:
print("amx.get_total_time()", amx.get_total_time())

In [ ]:
print("amx.get_fw_version()", amx.get_fw_version())

In [ ]:
print("amx.get_fw_date()", amx.get_fw_date())

In [ ]:
print("amx.get_hw_type()", amx.get_hw_type())

In [ ]:
print("amx.get_hw_version()", amx.get_hw_version())

In [ ]:
print("amx.get_product_id()", amx.get_product_id())

In [ ]:
print("amx.get_product_no()", amx.get_product_no())

In [ ]:
print("amx.get_config_list()", amx.get_config_list())

In [ ]:
print("amx.get_config_name(CONFIG_NUMBER)", amx.get_config_name(CONFIG_NUMBER))

In [ ]:
print("amx.get_config_flags(CONFIG_NUMBER)", amx.get_config_flags(CONFIG_NUMBER))

In [ ]:
print("amx.get_device_enable()", amx.get_device_enable())

In [ ]:
print("amx.get_device_enabled()", amx.get_device_enabled())

In [ ]:
print("amx.collect_housekeeping()", amx.collect_housekeeping())

In [ ]:
print("amx.disconnect()", amx.disconnect())

In [ ]:
amx.close()

## 3. Configured Session

Use a fresh initialized instance for timing and pulser helpers.
Provide a validated operating config first if your workflow depends on one.
Without an explicit slot number, `initialize()` will try to preload a valid
`Standby` config into runtime memory. Run cleanup cells at the end of the
section before reconnecting elsewhere.

AMX switch coarse delays are limited to `0..15` ticks for trigger rise,
trigger fall, and enable delay. The wrapper rejects out-of-range values
before they reach the vendor DLL.


In [ ]:
amx_oper = AMX(
    device_id=f"{DEVICE_ID}_oper",
    com=COM_PORT,
    port=PORT_NUMBER,
    baudrate=BAUDRATE,
)
oper_startup_kwargs = {"timeout_s": TIMEOUT_S}
if OPERATING_CONFIG_NUMBER is not None:
    oper_startup_kwargs["operating_config"] = OPERATING_CONFIG_NUMBER
if STANDBY_CONFIG_NUMBER is not None:
    oper_startup_kwargs["standby_config"] = STANDBY_CONFIG_NUMBER
oper_startup_state = amx_oper.initialize(**oper_startup_kwargs)

In [ ]:
print("oper_startup_state", oper_startup_state)

In [ ]:
print("amx_oper.get_oscillator_period()", amx_oper.get_oscillator_period())

In [ ]:
print("amx_oper.get_frequency_hz()", amx_oper.get_frequency_hz())

In [ ]:
print("amx_oper.set_frequency_hz(FREQUENCY_HZ)", amx_oper.set_frequency_hz(FREQUENCY_HZ))

In [ ]:
print("amx_oper.set_frequency_khz(FREQUENCY_KHZ)", amx_oper.set_frequency_khz(FREQUENCY_KHZ))

In [ ]:
print("amx_oper.set_device_enabled(True)", amx_oper.set_device_enabled(True))

In [ ]:
print("amx_oper.get_device_enabled()", amx_oper.get_device_enabled())

In [ ]:
print("amx_oper.get_pulser_delay(PULSER_NO)", amx_oper.get_pulser_delay(PULSER_NO))

In [ ]:
print("amx_oper.get_pulser_delay_ticks(PULSER_NO)", amx_oper.get_pulser_delay_ticks(PULSER_NO))

In [ ]:
print("amx_oper.get_pulser_delay_seconds(PULSER_NO)", amx_oper.get_pulser_delay_seconds(PULSER_NO))

In [ ]:
print("amx_oper.set_pulser_delay_ticks(PULSER_NO, PULSER_DELAY_TICKS)", amx_oper.set_pulser_delay_ticks(PULSER_NO, PULSER_DELAY_TICKS))

In [ ]:
print("amx_oper.get_pulser_width(PULSER_NO)", amx_oper.get_pulser_width(PULSER_NO))

In [ ]:
print("amx_oper.get_pulser_width_ticks(PULSER_NO)", amx_oper.get_pulser_width_ticks(PULSER_NO))

In [ ]:
print("amx_oper.get_pulser_width_seconds(PULSER_NO)", amx_oper.get_pulser_width_seconds(PULSER_NO))

In [ ]:
print("amx_oper.set_pulser_width_ticks(PULSER_NO, PULSER_WIDTH_TICKS)", amx_oper.set_pulser_width_ticks(PULSER_NO, PULSER_WIDTH_TICKS))

In [ ]:
print("amx_oper.set_pulser_duty_cycle(PULSER_NO, DUTY_CYCLE)", amx_oper.set_pulser_duty_cycle(PULSER_NO, DUTY_CYCLE))

In [ ]:
print("amx_oper.get_pulser_burst(PULSER_NO)", amx_oper.get_pulser_burst(PULSER_NO))

In [ ]:
print("amx_oper.get_switch_trigger_config(SWITCH_NO)", amx_oper.get_switch_trigger_config(SWITCH_NO))

In [ ]:
print("amx_oper.get_switch_enable_config(SWITCH_NO)", amx_oper.get_switch_enable_config(SWITCH_NO))

In [ ]:
print("amx_oper.get_switch_trigger_delay(SWITCH_NO)", amx_oper.get_switch_trigger_delay(SWITCH_NO))

In [ ]:
print("amx_oper.set_switch_trigger_delay(SWITCH_NO, SWITCH_TRIGGER_RISE_TICKS, SWITCH_TRIGGER_FALL_TICKS)", amx_oper.set_switch_trigger_delay(SWITCH_NO, SWITCH_TRIGGER_RISE_TICKS, SWITCH_TRIGGER_FALL_TICKS))

In [ ]:
print("amx_oper.get_switch_enable_delay(SWITCH_NO)", amx_oper.get_switch_enable_delay(SWITCH_NO))

In [ ]:
print("amx_oper.set_switch_enable_delay(SWITCH_NO, SWITCH_ENABLE_DELAY_TICKS)", amx_oper.set_switch_enable_delay(SWITCH_NO, SWITCH_ENABLE_DELAY_TICKS))

### Optional / Disruptive AMX Calls

The cells below intentionally change controller state more aggressively.
Uncomment them only when that behavior is desired on the hardware under test.


In [ ]:
# print("amx_oper.save_current_config(CONFIG_NUMBER)", amx_oper.save_current_config(CONFIG_NUMBER))

In [ ]:
amx_oper.shutdown()

In [ ]:
amx_oper.close()

## 4. Low-Level Transport Session

Dedicated instance for the explicit transport primitives. Do not mix this
section with `connect()` on the same object.


In [ ]:
amx_low = AMX(
    device_id=f"{DEVICE_ID}_low",
    com=COM_PORT,
    port=PORT_NUMBER,
    baudrate=BAUDRATE,
)

In [ ]:
print("amx_low.open_port(COM_PORT, PORT_NUMBER)", amx_low.open_port(COM_PORT, PORT_NUMBER))

In [ ]:
print("amx_low.set_baud_rate(BAUDRATE)", amx_low.set_baud_rate(BAUDRATE))

In [ ]:
print("amx_low.purge()", amx_low.purge())

In [ ]:
print("amx_low.device_purge()", amx_low.device_purge())

In [ ]:
print("amx_low.get_buffer_state()", amx_low.get_buffer_state())

In [ ]:
print("amx_low.load_current_config(CONFIG_NUMBER)", amx_low.load_current_config(CONFIG_NUMBER))

In [ ]:
print("amx_low.get_device_enable()", amx_low.get_device_enable())

In [ ]:
device_enable_status, current_enable = amx_low.get_device_enable()

In [ ]:
print("amx_low.set_device_enable(current_enable)", amx_low.set_device_enable(current_enable))

In [ ]:
osc_status, current_period = amx_low.get_oscillator_period()

In [ ]:
print("amx_low.set_oscillator_period(current_period)", amx_low.set_oscillator_period(current_period))

In [ ]:
delay_status, current_delay = amx_low.get_pulser_delay(PULSER_NO)

In [ ]:
print("amx_low.set_pulser_delay(PULSER_NO, current_delay)", amx_low.set_pulser_delay(PULSER_NO, current_delay))

In [ ]:
width_status, current_width = amx_low.get_pulser_width(PULSER_NO)

In [ ]:
print("amx_low.set_pulser_width(PULSER_NO, current_width)", amx_low.set_pulser_width(PULSER_NO, current_width))

In [ ]:
print("amx_low.close_port()", amx_low.close_port())

In [ ]:
amx_low.close()